In [0]:
import os

class DatabricksHelpers:
    def __init__(self, dbutils):
        self.dbutils = dbutils
        self._user = self._get_user()
        self._proj_root = self._get_project_root()

    def _get_user(self) -> str:
        """Extract current user email from dbutils context or fallback to Spark SQL."""
        try:
            context = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            tags = context.tags()
            return str(tags.apply("user"))
        except Exception:
            return spark.sql("SELECT current_user()").collect()[0][0]

    def _get_project_root(self) -> str:
        """Derives the project root directory relative to executing notebook location."""
        try:
            notebook_path = self.dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
            notebook_dir = os.path.dirname(f"/Workspace{notebook_path}")
            # Moves up from current directory to project root
            return os.path.normpath(os.path.join(notebook_dir, "../../"))
        except Exception:
            return os.getcwd()

    def current_user(self) -> str:
        """Returns the active user email."""
        return self._user

    def project_root(self) -> str:
        """Returns the absolute path to the project root directory."""
        return self._proj_root

In [0]:
# Instantiate helper
test_helpers = DatabricksHelpers(dbutils)

resolved_user = test_helpers.current_user()
resolved_root = test_helpers.project_root()

# ASSERTION 1: Verify current user format
assert isinstance(resolved_user, str) and "@" in resolved_user, (
    f"❌ TEST FAILED: Invalid user email resolved: '{resolved_user}'"
)

# ASSERTION 2: Verify project root exists on workspace disk
assert os.path.exists(resolved_root), (
    f"❌ TEST FAILED: Derived project root path does not exist on disk: '{resolved_root}'"
)

# ASSERTION 3: Verify no temporary exercise folders were created
assert not hasattr(test_helpers, "clean_working_directory"), (
    "❌ TEST FAILED: 'clean_working_directory' method should no longer exist!"
)

print(f"✓ TEST PASSED: DatabricksHelpers verified cleanly without creating workspace folders!")
print(f"  • User:         {resolved_user}")
print(f"  • Project Root: {resolved_root}")